# See how rounding changes a sum

Group the same three values in two orders and check their float32 results.

**Your exported case** · Near the precision limit

**Predict for the starter case.** For a = 16,777,216, b = 1, and c = −16,777,216, what happens to the 1 when you add a and b first? What if you add a and c first?

Run **Set up this notebook** once per kernel session. Then run **Your exported program**, or go straight to a numbered experiment. Each program cell contains its own kernel, inputs and launch. Shift+Enter runs that cell and checks that run’s returned array. The first line names the lesson, case and task. Leave it in place when editing Python below it.

Each numbered task opens with its own starting code, including any required earlier edits. To switch cases, run the cell under that case’s heading. To keep two versions, duplicate the cell before editing. Output belongs to the last execution of that cell. Rerun after each edit.

## Set up this notebook

Use a Python 3 IPython kernel in Jupyter or Colab. Reading the file and inspecting the references work offline. Executing CUDA programs needs an NVIDIA GPU and a working native backend. In Colab, choose **Runtime → Change runtime type → GPU** if one is available. For local Linux or Jetson, select the kernel for your CUDA environment.

Run the setup cell below to see the Python, architecture, compiler, driver, runtime and device it detects. For a fresh compatible CUDA 12 environment, review the package plan first in a separate cell:

```python
%pip install --dry-run "numba-cuda-mlir[cu12]==0.6.0" "numpy>=1.26,<2.5"
```

If the plan fits your environment, repeat without `--dry-run`, restart the kernel and rerun setup. Installation needs package access or an offline wheelhouse. On Jetson, record JetPack and check the ARM64 wheels against your Python and system CUDA versions before installing. A compatible wheel alone does not establish device support.

To recover a starting program, run `print(lab_start("CASE", "TASK"))` in a new cell, using the two IDs after the lesson name on its first line. Copy the printed Python into a duplicate of the program cell. Your original edits stay in place.

In [ ]:
# Register the offline experiment runner and check this environment.
"""Native lesson support embedded in each exported IPython notebook."""
import contextlib
import hashlib
import io
import json
import linecache
import platform
import re
import sys
from importlib import metadata
from pathlib import Path


def lab_preflight():
    """Report the running environment. No install, kernel launch or backend substitution."""
    print(f"Python {platform.python_version()} · {platform.platform()} · {sys.executable}")
    jetpack = Path("/etc/nv_tegra_release")
    if jetpack.is_file():
        print("Jetson Linux:", jetpack.read_text().splitlines()[0])
    versions = {}
    for package in ("numba-cuda-mlir", "numpy", "cuda-bindings", "cuda-toolkit", "nvidia-cuda-nvcc-cu12"):
        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            versions[package] = "not installed"
        print(f"{package}: {versions[package]}")
    try:
        from cuda.bindings import runtime
        status, runtime_version = runtime.cudaRuntimeGetVersion()
        print(f"CUDA runtime API: {runtime_version} ({status})")
        if int(status) or not 12000 <= runtime_version < 13000:
            raise RuntimeError("The pinned cu12 environment needs a working CUDA 12 runtime.")
        from cuda.bindings import driver
        status, driver_version = driver.cuDriverGetVersion()
        print(f"Driver CUDA API: {driver_version} ({status})")
        if int(status) or driver_version < 12000:
            raise RuntimeError("The pinned runtime needs a compatible NVIDIA driver.")
        from numba_cuda_mlir import cuda
        if versions["numba-cuda-mlir"] != "0.6.0":
            raise RuntimeError("This course pins numba-cuda-mlir 0.6.0. Check the installation plan above.")
        if not cuda.is_available():
            raise RuntimeError("The selected Python kernel cannot access an NVIDIA GPU.")
        device = cuda.get_current_device()
        print(f"Native device: {device.name!s}; compute capability {device.compute_capability}")
        if tuple(device.compute_capability) < (7, 0):
            raise RuntimeError("The pinned backend requires compute capability 7.0 or later.")
    except Exception as error:
        print(f"Native execution unavailable: {type(error).__name__}: {error}")
        print("Check GPU allocation, driver and kernel environment. On Jetson, check JetPack and the ARM64 package plan.")
        return False
    print("Device detected. Run a program cell to verify compilation and execution on this device.")
    return True


def _task(case, key):
    task = LAB["tasks"][key]
    if task["case"] != case:
        raise ValueError(f"Task {key} belongs to case {task['case']}, not {case}.")
    return task


def _apply_edits(source, ids):
    for edit in ids:
        pairs = LAB["edits"][edit]
        for old, new in zip(pairs[::2], pairs[1::2]):
            source = source.replace(old, new)
    return source


def lab_start(case, key):
    """Return a fresh copy of the supplied source without modifying any cell."""
    starter = LAB["cases"][case]["starter"]
    if key == "exported":
        if case != LAB["exportedCase"]:
            raise ValueError("Choose the case named above Your exported program.")
        return LAB["exported"]
    if key == "starter":
        return starter
    return _apply_edits(starter, _task(case, key)["before"])


def _target(case, key):
    task = _task(case, key)
    if task["restore"]:
        return LAB["cases"][case]["starter"]
    return _apply_edits(lab_start(case, key), task["after"])


def _compare(actual, expected, spec):
    """Compare without converting the learner's output to the reference dtype or shape."""
    import numpy as np
    ref = np.array(expected, dtype=spec["dtype"]).reshape(spec["shape"])
    shape_ok = isinstance(actual, np.ndarray) and actual.shape == ref.shape
    dtype_ok = getattr(actual, "dtype", None) == ref.dtype
    soft = LAB["lesson"] == "softmax"
    rtol, atol = (1e-5, 1e-7) if soft else (0, 0)
    print(f"  Shape: {getattr(actual, 'shape', None)}; expected {ref.shape} · {'match' if shape_ok else 'different'}")
    print(f"  Dtype: {getattr(actual, 'dtype', None)}; expected {ref.dtype} · {'match' if dtype_ok else 'different'}")
    if not shape_ok or actual.dtype.kind not in "fiu":
        print("  Finiteness and values: unavailable for this shape or element type.")
        return False
    # Only authored overflow locations may be NaN. Infinity is never a reference value.
    finite_ok = np.array_equal(np.isnan(actual), np.isnan(ref)) and not np.isinf(actual).any()
    print(f"  Finiteness: {int(np.isfinite(actual).sum())}/{ref.size} finite; expected NaNs {int(np.isnan(ref).sum())} · {'match' if finite_ok else 'different'}")
    close = np.isclose(actual, ref, rtol=rtol, atol=atol, equal_nan=True) if soft else np.equal(actual, ref)
    print(f"  Values: {int(close.sum())}/{ref.size} match (rtol={rtol:g}, atol={atol:g}).")
    if not close.all():
        index = tuple(int(i) for i in np.argwhere(~close)[0])
        print(f"  First difference at {index}: observed {actual[index]}, expected {ref[index]}.")
    return bool(shape_ok and dtype_ok and finite_ok and close.all())


def _check_prints(lines, references):
    """The authored host prints follow the primary result, in source order."""
    import numpy as np
    if not references:
        return True
    extra = lines[1:]
    if len(extra) != len(references):
        print(f"  Additional output: expected {len(references)} lines, observed {len(extra)}.")
        return False
    for line, reference in zip(extra, references):
        if isinstance(reference, str):
            match = line.strip() == reference
        else:
            tokens = re.findall(r"[-+]?(?:nan|inf|(?:\d+(?:\.\d*)?|\.\d+)(?:e[-+]?\d+)?)", line.lower())
            values = np.array([float(t) for t in tokens])
            expected = np.asarray(reference, dtype=float)
            atol = 2e-6 if LAB["lesson"] == "softmax" else 0
            match = values.shape == expected.shape and np.allclose(values, expected, atol=atol, rtol=0, equal_nan=True)
        if not match:
            print(f"  Additional output differs: {line!r}; expected {reference}.")
            return False
    print("  Additional host output matches the requested observation.")
    return True


def assess(case, key, source, actual, lines):
    import numpy as np
    spec = LAB["cases"][case]
    # IPython may supply one fewer terminal newline to a cell magic.
    body = source.rstrip("\n")
    starter = spec["starter"].rstrip("\n")
    print("Starter comparison")
    baseline = _compare(actual, spec["expected"], spec)
    task_key = key
    if key == "exported":
        selected = LAB["selectedTask"]
        candidates = [k for k, t in LAB["tasks"].items() if t["case"] == case and t["kind"] == "run"]
        if selected:
            task_key = selected
        elif body == starter:
            task_key = "starter"
        else:
            task_key = next((k for k in candidates if body == _target(case, k).rstrip("\n")), "starter")
    if task_key == "starter":
        if body != starter:
            print("Objective unassessed: this custom program has no matching authored reference.")
            return "unassessed"
        contract = {"policy":"probabilities" if LAB["lesson"] == "softmax" else "values", "values":spec["expected"], "prints":[]}
    else:
        t = _task(case, task_key)
        if t["kind"] == "reason":
            print("Source reasoning: explain the task using the supplied code.")
            return "reasoning"
        if body != _target(case, task_key).rstrip("\n"):
            if body == lab_start(case, task_key).rstrip("\n"):
                print("Starting program executed. Make the requested edit in this cell, then rerun to check its objective.")
                return "starting-point"
            print("Objective unassessed: the program differs from the authored edit for this task.")
            return "unassessed"
        contract = t["objective"]
    policy = contract["policy"]
    print(f"Objective · {task_key}")
    if policy == "race":
        valid = isinstance(actual, np.ndarray) and actual.dtype == np.dtype(spec["dtype"]) and list(actual.shape) == spec["shape"]
        if not valid:
            print("The observed counters have the wrong shape or dtype.")
            return "fail"
        hot = int(np.argmax(spec["expected"]))
        if np.any(np.delete(actual, hot) != 0) or not 1 <= actual[hot] <= spec["expected"][hot]:
            print("Counters fall outside this hot-bin experiment: only the routed bin receives increments.")
            return "fail"
        print(f"Observed counters: {actual.tolist()}. Repeat and record bin 2. Even an exact starter match cannot establish atomicity or race safety.")
        return "observation"
    ok = baseline if contract["values"] == spec["expected"] else _compare(actual, contract["values"], spec)
    if policy == "probabilities" and isinstance(actual, np.ndarray) and list(actual.shape) == spec["shape"] and actual.dtype.kind == "f":
        sums = actual.sum(axis=1, dtype=np.float64)
        print("  Probability row sums:", sums, "(absolute tolerance 2e-6); each probability must be in [0, 1].")
        ok = ok and bool(((actual >= 0) & (actual <= 1)).all() and np.allclose(sums, 1, atol=2e-6, rtol=0))
    if policy == "overflow":
        print("  The unshifted exponential overflows or underflows on these inputs. Compare the NaN locations above.")
    if policy == "precision" and isinstance(actual, np.ndarray) and list(actual.shape) == spec["shape"] and actual.dtype.kind in "fiu":
        exact = np.array(contract["values"])
        print("  Absolute error versus the mathematical sum: starter", abs(np.array(spec["expected"]) - exact), "this run", abs(actual - exact))
    prints_ok = _check_prints(lines, contract["prints"])
    verdict = "pass" if ok and prints_ok else "fail"
    print(f"Numerical objective: {verdict}. Explain the source or measurements requested in the task.")
    return verdict


def execute_program(source):
    """IPython registry-independent execution, with source available for JIT inspection."""
    filename = "<cuda-lab-" + hashlib.sha256(source.encode()).hexdigest() + ">"
    linecache.cache[filename] = (len(source), None, source.splitlines(keepends=True), filename)
    scope = {"__name__": "__main__", "__file__": filename}
    output = io.StringIO()
    try:
        with contextlib.redirect_stdout(output):
            exec(compile(source, filename, "exec"), scope)
    finally:
        print(output.getvalue(), end="")
    return scope.get("actual"), output.getvalue().splitlines()


def run_experiment(line, cell):
    lesson, case, key = line.split()
    if lesson != LAB["lesson"]:
        raise ValueError("Run the setup cell from this notebook before its program cells.")
    lab_start(case, key)  # Validate the fixture/task binding before executing user code.
    print(f"{lesson} / {case} / {key} · source SHA-256 prefix {hashlib.sha256(cell.encode()).hexdigest()[:16]}")
    if not NATIVE_READY:
        raise RuntimeError("Native preflight did not pass. Review setup and rerun it. No result was checked.")
    try:
        actual, lines = execute_program(cell)
    except Exception:
        print("Program execution failed. No result was checked.")
        raise
    if actual is None:
        print("Objective unassessed: this run did not assign a returned array to actual.")
        return
    assess(case, key, cell, actual, lines)

LAB = json.loads("{\"lesson\":\"precision\",\"exportedCase\":\"near\",\"exported\":\"from numba_cuda_mlir import cuda\\nfrom numba_cuda_mlir.types import float32\\n\\n\\n@cuda.jit\\ndef kernel(a, b, c, out, n):\\n    i = cuda.grid(1)\\n    if i < n:\\n        first = float32(a[i] + c[i])\\n        out[i] = float32(first + b[i])\\n\\n\\n# Host code\\ndef main():\\n    import numpy as np\\n    from numba_cuda_mlir import cuda\\n\\n    a_host = np.array([16777216, 8388608], dtype=np.float32)\\n    a = cuda.to_device(a_host)\\n\\n    b_host = np.array([1, 1], dtype=np.float32)\\n    b = cuda.to_device(b_host)\\n\\n    c_host = np.array([-16777216, -8388608], dtype=np.float32)\\n    c = cuda.to_device(c_host)\\n\\n    out_host = np.zeros(2, dtype=np.float32)\\n    out = cuda.to_device(out_host)\\n\\n    kernel[1, 4](a, b, c, out, 2)\\n    result = out.copy_to_host()\\n    print(\\\"out =\\\", result.tolist())\\n    return result\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    actual = main()\\n\",\"selectedTask\":\"regroup\",\"cases\":{\"near\":{\"label\":\"Near the precision limit\",\"shape\":[2],\"dtype\":\"float32\",\"expected\":[0,1],\"starter\":\"from numba_cuda_mlir import cuda\\nfrom numba_cuda_mlir.types import float32\\n\\n\\n@cuda.jit\\ndef kernel(a, b, c, out, n):\\n    i = cuda.grid(1)\\n    if i < n:\\n        first = float32(a[i] + b[i])\\n        out[i] = float32(first + c[i])\\n\\n\\n# Host code\\ndef main():\\n    import numpy as np\\n    from numba_cuda_mlir import cuda\\n\\n    a_host = np.array([16777216, 8388608], dtype=np.float32)\\n    a = cuda.to_device(a_host)\\n\\n    b_host = np.array([1, 1], dtype=np.float32)\\n    b = cuda.to_device(b_host)\\n\\n    c_host = np.array([-16777216, -8388608], dtype=np.float32)\\n    c = cuda.to_device(c_host)\\n\\n    out_host = np.zeros(2, dtype=np.float32)\\n    out = cuda.to_device(out_host)\\n\\n    kernel[1, 4](a, b, c, out, 2)\\n    result = out.copy_to_host()\\n    print(\\\"out =\\\", result.tolist())\\n    return result\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    actual = main()\\n\"},\"small\":{\"label\":\"Smaller integers\",\"shape\":[2],\"dtype\":\"float32\",\"expected\":[1,1],\"starter\":\"from numba_cuda_mlir import cuda\\nfrom numba_cuda_mlir.types import float32\\n\\n\\n@cuda.jit\\ndef kernel(a, b, c, out, n):\\n    i = cuda.grid(1)\\n    if i < n:\\n        first = float32(a[i] + b[i])\\n        out[i] = float32(first + c[i])\\n\\n\\n# Host code\\ndef main():\\n    import numpy as np\\n    from numba_cuda_mlir import cuda\\n\\n    a_host = np.array([100, 1000], dtype=np.float32)\\n    a = cuda.to_device(a_host)\\n\\n    b_host = np.array([1, 1], dtype=np.float32)\\n    b = cuda.to_device(b_host)\\n\\n    c_host = np.array([-100, -1000], dtype=np.float32)\\n    c = cuda.to_device(c_host)\\n\\n    out_host = np.zeros(2, dtype=np.float32)\\n    out = cuda.to_device(out_host)\\n\\n    kernel[1, 4](a, b, c, out, 2)\\n    result = out.copy_to_host()\\n    print(\\\"out =\\\", result.tolist())\\n    return result\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    actual = main()\\n\"}},\"tasks\":{\"round\":{\"case\":\"near\",\"kind\":\"run\",\"before\":[],\"after\":[],\"restore\":false,\"objective\":{\"values\":[0,1],\"policy\":\"values\",\"prints\":[]}},\"regroup\":{\"case\":\"near\",\"kind\":\"run\",\"before\":[],\"after\":[\"regroup\"],\"restore\":false,\"objective\":{\"values\":[1,1],\"policy\":\"precision\",\"prints\":[]}},\"small\":{\"case\":\"small\",\"kind\":\"run\",\"before\":[],\"after\":[],\"restore\":false,\"objective\":{\"values\":[1,1],\"policy\":\"values\",\"prints\":[]}}},\"edits\":{\"regroup\":[\"a[i] + b[i]\",\"a[i] + c[i]\",\"first + c[i]\",\"first + b[i]\"]}}")
get_ipython().register_magic_function(run_experiment, "cell", "experiment")
NATIVE_READY = lab_preflight()


## Your exported program

**Near the precision limit** · `near`

Selected task at export: **Change the grouping**. This copy contains the open editor text, including host changes. Recognized legacy CUDA imports are adapted for the native backend. The numbered experiments and starter cases use course source. Other saved case edits stay in your browser. Use **Editor tools → Export all learner work** there for a full backup.

The checker compares a recognized program with its case reference. For custom code it shows a starter comparison and leaves the custom objective unassessed. Shape and dtype must match exactly. These small integer or exactly representable float32 fixture results use zero absolute and relative tolerance. To supply a result, assign the returned NumPy array to `actual` in the program.

In [ ]:
%%experiment precision near exported
from numba_cuda_mlir import cuda
from numba_cuda_mlir.types import float32


@cuda.jit
def kernel(a, b, c, out, n):
    i = cuda.grid(1)
    if i < n:
        first = float32(a[i] + c[i])
        out[i] = float32(first + b[i])


# Host code
def main():
    import numpy as np
    from numba_cuda_mlir import cuda

    a_host = np.array([16777216, 8388608], dtype=np.float32)
    a = cuda.to_device(a_host)

    b_host = np.array([1, 1], dtype=np.float32)
    b = cuda.to_device(b_host)

    c_host = np.array([-16777216, -8388608], dtype=np.float32)
    c = cuda.to_device(c_host)

    out_host = np.zeros(2, dtype=np.float32)
    out = cuda.to_device(out_host)

    kernel[1, 4](a, b, c, out, 2)
    result = out.copy_to_host()
    print("out =", result.tolist())
    return result


if __name__ == "__main__":
    actual = main()


## 1. Observe float32 rounding

**Starter case** · Near the precision limit

**Predict for the starter case.** For a = 16,777,216, b = 1, and c = −16,777,216, what happens to the 1 when you add a and b first? What if you add a and c first?

Start with the supplied case program. Run Near the precision limit. Thread 0 starts with 16,777,216 and Thread 1 starts with 8,388,608. Both add 1 and then subtract their starting value. Compare the two outputs.

**Evidence to examine.** Compare both outputs with the exact sum of one.

In [ ]:
%%experiment precision near round
from numba_cuda_mlir import cuda
from numba_cuda_mlir.types import float32


@cuda.jit
def kernel(a, b, c, out, n):
    i = cuda.grid(1)
    if i < n:
        first = float32(a[i] + b[i])
        out[i] = float32(first + c[i])


# Host code
def main():
    import numpy as np
    from numba_cuda_mlir import cuda

    a_host = np.array([16777216, 8388608], dtype=np.float32)
    a = cuda.to_device(a_host)

    b_host = np.array([1, 1], dtype=np.float32)
    b = cuda.to_device(b_host)

    c_host = np.array([-16777216, -8388608], dtype=np.float32)
    c = cuda.to_device(c_host)

    out_host = np.zeros(2, dtype=np.float32)
    out = cuda.to_device(out_host)

    kernel[1, 4](a, b, c, out, 2)
    result = out.copy_to_host()
    print("out =", result.tolist())
    return result


if __name__ == "__main__":
    actual = main()


## 2. Change the grouping

**Starter case** · Near the precision limit

**Predict for the starter case.** For a = 16,777,216, b = 1, and c = −16,777,216, what happens to the 1 when you add a and b first? What if you add a and c first?

Start with the supplied case program. In the kernel, change a[i] + b[i] to a[i] + c[i] and first + c[i] to first + b[i]. Rerun and explain why only one result changes.

**Evidence to examine.** Identify which thread’s output changes.

In [ ]:
%%experiment precision near regroup
from numba_cuda_mlir import cuda
from numba_cuda_mlir.types import float32


@cuda.jit
def kernel(a, b, c, out, n):
    i = cuda.grid(1)
    if i < n:
        first = float32(a[i] + b[i])
        out[i] = float32(first + c[i])


# Host code
def main():
    import numpy as np
    from numba_cuda_mlir import cuda

    a_host = np.array([16777216, 8388608], dtype=np.float32)
    a = cuda.to_device(a_host)

    b_host = np.array([1, 1], dtype=np.float32)
    b = cuda.to_device(b_host)

    c_host = np.array([-16777216, -8388608], dtype=np.float32)
    c = cuda.to_device(c_host)

    out_host = np.zeros(2, dtype=np.float32)
    out = cuda.to_device(out_host)

    kernel[1, 4](a, b, c, out, 2)
    result = out.copy_to_host()
    print("out =", result.tolist())
    return result


if __name__ == "__main__":
    actual = main()


## 3. Check the original grouping

**Starter case** · Smaller integers

**Predict for the starter case.** For a = 100 and 1000, does float32 preserve the added 1 before subtracting a?

Start with the supplied case program. Run Smaller integers with the starter. Check whether the original grouping preserves 1 for both values.

**Evidence to examine.** Check the first addition in the source as well as both outputs.

In [ ]:
%%experiment precision small small
from numba_cuda_mlir import cuda
from numba_cuda_mlir.types import float32


@cuda.jit
def kernel(a, b, c, out, n):
    i = cuda.grid(1)
    if i < n:
        first = float32(a[i] + b[i])
        out[i] = float32(first + c[i])


# Host code
def main():
    import numpy as np
    from numba_cuda_mlir import cuda

    a_host = np.array([100, 1000], dtype=np.float32)
    a = cuda.to_device(a_host)

    b_host = np.array([1, 1], dtype=np.float32)
    b = cuda.to_device(b_host)

    c_host = np.array([-100, -1000], dtype=np.float32)
    c = cuda.to_device(c_host)

    out_host = np.zeros(2, dtype=np.float32)
    out = cuda.to_device(out_host)

    kernel[1, 4](a, b, c, out, 2)
    result = out.copy_to_host()
    print("out =", result.tolist())
    return result


if __name__ == "__main__":
    actual = main()


## Explain the authored examples

The first case starts with a values of 16,777,216 and 8,388,608. The original order returns 0 for the first thread and 1 for the second. Canceling a and c before adding b gives 1 for both. The starter reference follows the original float32 grouping. The exact sum is 1 for both inputs, so the edited result at index 0 is closer to the exact sum.

### Optional native investigation

In a separate Python cell, run import numpy as np and print(np.float64(16777216) + 1 - 16777216). Compare the value with the first output of the original float32 grouping.